# Fashion Product Search: Text-Based Product Discovery

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SerendipityOneInc/APIClaw-Notebook/blob/main/fashion/fashion_product_search_demo.ipynb)

This notebook demonstrates how to use APIClaw's **Fashion Product Search** API to discover fashion products across 200M+ items from major retailers using natural-language text queries.

### What you'll learn

1. **Basic text search** — find products with natural-language queries
2. **Brand filtering** — narrow results to specific brands
3. **Price range filtering** — search within a budget
4. **Retailer filtering** — allowlist or denylist specific retailer domains
5. **Building a product comparison table** — aggregate results across queries

### API Reference

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/openapi/v2/model/fashion-product-search` | POST | Search fashion products by text query |

## Setup

Get a free API key (1,000 credits) at [apiclaw.io/register](https://apiclaw.io/register).

In [ ]:
!pip install -q httpx Pillow matplotlib

In [ ]:
import getpass
import os

if "APICLAW_API_KEY" not in os.environ:
    os.environ["APICLAW_API_KEY"] = getpass.getpass("Enter your APIClaw API key (hms_live_...): ")

API_KEY = os.environ["APICLAW_API_KEY"]
BASE_URL = "https://api.apiclaw.io/openapi/v2"

In [ ]:
import httpx
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

client = httpx.Client(
    base_url=BASE_URL,
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=30.0,
)


def fashion_search(query: str, top_k: int = 10, **kwargs) -> list[dict]:
    """Search fashion products by text query."""
    payload = {"query": query, "topK": top_k, **kwargs}
    resp = client.post("/model/fashion-product-search", json=payload)
    resp.raise_for_status()
    return resp.json()["data"]["products"]


def load_image(url: str) -> Image.Image:
    """Download an image from URL."""
    r = httpx.get(url, timeout=10, follow_redirects=True)
    return Image.open(BytesIO(r.content))


def show_products(products: list[dict], max_cols: int = 5):
    """Display product images in a grid."""
    n = min(len(products), max_cols)
    fig, axes = plt.subplots(1, n, figsize=(3.5 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, p in zip(axes, products[:n]):
        try:
            ax.imshow(load_image(p.get("imageUrl", "")))
        except Exception:
            ax.text(0.5, 0.5, "N/A", ha="center", va="center")
        title = (p.get("title") or "Unknown")[:40]
        price = p.get("price")
        brand = p.get("brand") or ""
        label = f"{title}...\n{brand}" + (f" | ${price:.2f}" if price else "")
        ax.set_title(label, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

---
## Task 1: Basic Text Search

The simplest use case — type a natural-language query and get matching products from 200M+ fashion items across major retailers.

In [ ]:
# Simple natural-language search
results = fashion_search("women red floral midi dress", top_k=5)

print(f"Found {len(results)} products\n")
for p in results:
    print(f"  {p.get('title', 'N/A')[:60]}")
    print(f"    Brand: {p.get('brand', '-')} | Price: ${p.get('price', 0):.2f} | Site: {p.get('site', '-')}")
    print()

show_products(results)

In [ ]:
# Try different query styles
queries = [
    "black leather ankle boots",
    "men slim fit navy blazer",
    "women brown suede tote under 300",
    "summer linen shorts casual",
]

for q in queries:
    products = fashion_search(q, top_k=3)
    print(f"\n\"{q}\" → {len(products)} results")
    for p in products:
        print(f"  {p.get('title', '')[:55]} | ${p.get('price', 0):.2f} | {p.get('site', '')}")

---
## Task 2: Brand Filtering

Narrow results to a specific brand. Useful for brand-specific product research, competitive benchmarking, or building brand-focused shopping assistants.

In [ ]:
# Search for handbags from a specific brand
coach_bags = fashion_search("leather tote bag", top_k=5, brand="Coach")

print(f"Coach leather tote bags: {len(coach_bags)} results\n")
for p in coach_bags:
    print(f"  {p.get('title', '')[:55]}")
    print(f"    ${p.get('price', 0):.2f} | {p.get('site', '')}")

show_products(coach_bags)

In [ ]:
# Compare the same product category across brands
brands = ["Nike", "Adidas", "New Balance"]
brand_results = {}

for brand in brands:
    products = fashion_search("running shoes", top_k=5, brand=brand)
    brand_results[brand] = products
    avg_price = sum(p.get("price", 0) for p in products) / max(len(products), 1)
    print(f"{brand}: {len(products)} results, avg price ${avg_price:.2f}")

# Show top result from each brand side by side
top_picks = [products[0] for products in brand_results.values() if products]
show_products(top_picks)

---
## Task 3: Price Range Filtering

Filter results by price range (in USD). Useful for budget-aware recommendations or tiered product catalogs.

In [ ]:
# Search within a budget
budget_dresses = fashion_search(
    "elegant evening dress",
    top_k=5,
    priceMin=50,
    priceMax=200,
)

print(f"Evening dresses $50-$200: {len(budget_dresses)} results\n")
for p in budget_dresses:
    print(f"  ${p.get('price', 0):>7.2f} | {p.get('title', '')[:50]} | {p.get('site', '')}")

show_products(budget_dresses)

In [ ]:
# Price tier analysis: same query across budget / mid / premium
tiers = [
    ("Budget",  0, 50),
    ("Mid-range", 50, 200),
    ("Premium", 200, 1000),
]

print("Price tier analysis for 'women handbag':\n")
for label, lo, hi in tiers:
    products = fashion_search("women handbag", top_k=10, priceMin=lo, priceMax=hi)
    prices = [p.get("price", 0) for p in products if p.get("price")]
    if prices:
        print(f"  {label:>10} (${lo}-${hi}): {len(products)} results | "
              f"avg ${sum(prices)/len(prices):.2f} | "
              f"range ${min(prices):.2f}-${max(prices):.2f}")
    else:
        print(f"  {label:>10} (${lo}-${hi}): no results")

---
## Task 4: Retailer Filtering

Use `sites` (allowlist) or `excludeSites` (denylist) to control which retailer domains appear in results. Useful for affiliate programs, marketplace comparison, or excluding resale platforms.

In [ ]:
# Only search on specific retailers
luxury_sites = fashion_search(
    "designer sunglasses",
    top_k=5,
    sites=["farfetch.com", "nordstrom.com", "net-a-porter.com"],
)

print(f"Designer sunglasses from luxury retailers: {len(luxury_sites)} results\n")
for p in luxury_sites:
    print(f"  {p.get('site', ''):>20} | ${p.get('price', 0):.2f} | {p.get('title', '')[:45]}")

In [ ]:
# Exclude resale / secondhand platforms
new_only = fashion_search(
    "vintage style denim jacket",
    top_k=5,
    excludeSites=["poshmark.com", "therealreal.com", "ebay.com"],
)

print(f"Denim jackets (excl. resale): {len(new_only)} results\n")
for p in new_only:
    print(f"  {p.get('site', ''):>20} | ${p.get('price', 0):.2f} | {p.get('title', '')[:45]}")

show_products(new_only)

---
## Task 5: Building a Product Comparison Table

Combine multiple searches to build a comparison dataset. This pattern is useful for:
- Competitive pricing analysis across brands
- Trend spotting (which styles are most available)
- Building product recommendation datasets

In [ ]:
# Build a comparison dataset across categories
categories = {
    "Dresses": "women summer dress",
    "Sneakers": "casual white sneakers",
    "Bags": "leather crossbody bag",
    "Jackets": "men bomber jacket",
}

all_products = []
for cat_name, query in categories.items():
    products = fashion_search(query, top_k=10)
    for p in products:
        all_products.append({
            "category": cat_name,
            "title": (p.get("title") or "")[:60],
            "brand": p.get("brand", "-"),
            "price": p.get("price", 0),
            "site": p.get("site", "-"),
            "rating": p.get("rating"),
            "score": p.get("score"),
        })

print(f"Total products collected: {len(all_products)}")

In [ ]:
# Summary statistics by category
print(f"{'Category':>12} | {'Count':>5} | {'Avg Price':>9} | {'Min':>7} | {'Max':>7} | Top Brands")
print("-" * 80)

for cat_name in categories:
    cat_products = [p for p in all_products if p["category"] == cat_name]
    prices = [p["price"] for p in cat_products if p["price"]]
    brands = [p["brand"] for p in cat_products if p["brand"] != "-"]
    # Count brand frequency
    brand_counts = {}
    for b in brands:
        brand_counts[b] = brand_counts.get(b, 0) + 1
    top_brands = sorted(brand_counts, key=brand_counts.get, reverse=True)[:3]

    if prices:
        print(f"{cat_name:>12} | {len(cat_products):>5} | "
              f"${sum(prices)/len(prices):>7.2f} | "
              f"${min(prices):>5.2f} | ${max(prices):>5.2f} | "
              f"{', '.join(top_brands)}")

In [ ]:
# Visualize price distribution by category
fig, ax = plt.subplots(figsize=(10, 5))

cat_names = list(categories.keys())
cat_prices = []
for cat in cat_names:
    prices = [p["price"] for p in all_products if p["category"] == cat and p["price"]]
    cat_prices.append(prices)

ax.boxplot(cat_prices, labels=cat_names)
ax.set_ylabel("Price (USD)")
ax.set_title("Price Distribution by Fashion Category")
plt.tight_layout()
plt.show()

---
## Summary

This notebook demonstrated how to use APIClaw's Fashion Product Search API for text-based fashion product discovery.

### API Used

| Endpoint | Purpose | Cost |
|----------|---------|------|
| `POST /openapi/v2/model/fashion-product-search` | Text-based fashion product search | 1 credit/request |

### Key Parameters

| Parameter | Description |
|-----------|-------------|
| `query` | Natural-language search query |
| `topK` | Max results (1-50) |
| `brand` | Filter by brand name |
| `priceMin` / `priceMax` | Price range in USD |
| `sites` | Retailer domain allowlist |
| `excludeSites` | Retailer domain denylist |

### Next Steps

- Try the [Fashion Image Search](https://colab.research.google.com/github/SerendipityOneInc/APIClaw-Notebook/blob/main/fashion/fashion_image_search_demo.ipynb) notebook for visual similarity search
- Combine text search with [FashionSigLIP2 embeddings](https://colab.research.google.com/github/SerendipityOneInc/APIClaw-Notebook/blob/main/fashion/fashion_siglip2_demo.ipynb) for custom reranking
- Get your API key at [apiclaw.io](https://apiclaw.io)